# Build an AI Agent That Never Forgets — Oracle AI Database 26ai

This notebook demonstrates how to build a LangChain agent with persistent memory backed by Oracle AI Database 26ai.

**What you will build:**
- A standard LangChain agent without memory (the problem)
- The same agent with Oracle 26ai as the unified memory backend (the fix)

**Prerequisites:**
- Oracle AI Database 26ai running via Docker
- OpenAI API key set as `OPENAI_API_KEY` environment variable
- Python packages: `langchain`, `langchain-oracledb`, `oracledb`, `openai`, `langchain-openai`, `langchain-classic`

**Setup:**
```bash
docker run -d --name oracle-26ai -p 1521:1521 -e ORACLE_PWD=ByteMonk123 container-registry.oracle.com/database/free:latest
pip install langchain langchain-oracledb oracledb openai langchain-openai langchain-classic
```

## Part 1 — The Problem: Agent Without Memory

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_openai_functions_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool

llm = ChatOpenAI(model="gpt-4o", temperature=0)

@tool
def get_current_time() -> str:
    """Returns the current time."""
    from datetime import datetime
    return datetime.now().strftime("%H:%M:%S")

tools = [get_current_time]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent    = create_openai_functions_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools)

In [ ]:
# Session 1 — introduce myself
r1 = executor.invoke({"input": "My name is Himalay. I am a software engineer and founder of ByteMonk."})
print(r1["output"])

In [ ]:
# Session 2 — new invocation, no history passed in
r2 = executor.invoke({"input": "What do you know about me?"})
print(r2["output"])
# Output: "I don't have any information about you."

## Part 2 — The Fix: Oracle 26ai as the Memory Backend

In [ ]:
import oracledb
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_classic.agents import AgentExecutor, create_openai_functions_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_oracledb import OracleVS
from langchain_community.vectorstores.utils import DistanceStrategy

In [ ]:
connection = oracledb.connect(
    user="pdbadmin",
    password="ByteMonk123",
    dsn="localhost:1521/FREEPDB1"
)
cursor = connection.cursor()
print("Connected to Oracle 26ai")

In [ ]:
embeddings   = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = OracleVS(connection, embeddings, "AGENT_MEMORIES", DistanceStrategy.COSINE)
print("Vector store ready")

In [ ]:
cursor.execute("""
    DECLARE
        v_count NUMBER;
    BEGIN
        SELECT COUNT(*) INTO v_count FROM user_tables
        WHERE table_name = 'CONVERSATION_HISTORY';
        IF v_count = 0 THEN
            EXECUTE IMMEDIATE '
                CREATE TABLE CONVERSATION_HISTORY (
                    id         NUMBER GENERATED ALWAYS AS IDENTITY,
                    session_id VARCHAR2(100),
                    role       VARCHAR2(20),
                    content    CLOB,
                    created_at TIMESTAMP DEFAULT SYSTIMESTAMP
                )';
        END IF;
    END;
""")
connection.commit()
print("Tables ready")

In [ ]:
def save_memory(session_id, role, content):
    cursor.execute(
        "INSERT INTO CONVERSATION_HISTORY (session_id, role, content) VALUES (:1, :2, :3)",
        [session_id, role, content]
    )
    connection.commit()
    if role == "human":
        vector_store.add_texts(texts=[content], metadatas=[{"session_id": session_id}])

def load_memory(session_id, limit=10):
    cursor.execute(
        "SELECT role, content FROM CONVERSATION_HISTORY WHERE session_id = :1 ORDER BY created_at DESC FETCH FIRST :2 ROWS ONLY",
        [session_id, limit]
    )
    rows = cursor.fetchall()
    rows.reverse()
    return [(r, c.read() if hasattr(c, "read") else str(c)) for r, c in rows]

def search_memory(query, k=3):
    try:
        return [doc.page_content for doc in vector_store.similarity_search(query, k=k)]
    except Exception:
        return []

In [ ]:
llm   = ChatOpenAI(model="gpt-4o", temperature=0)
tools = [get_current_time]

def run_agent(user_input, session_id):
    history_rows      = load_memory(session_id)
    semantic_memories = search_memory(user_input)
    chat_history = []
    for role, content in history_rows:
        chat_history.append(("human" if role == "human" else "assistant", content))
    system_context = "You are a helpful assistant with memory."
    if semantic_memories:
        system_context += "\n\nRelevant context from past conversations:\n"
        system_context += "\n".join(f"- {m}" for m in semantic_memories)
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_context),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])
    agent    = create_openai_functions_agent(llm, tools, prompt)
    executor = AgentExecutor(agent=agent, tools=tools)
    response = executor.invoke({"input": user_input, "chat_history": chat_history})
    output   = response["output"]
    save_memory(session_id, "human",     user_input)
    save_memory(session_id, "assistant", output)
    return output

## Part 3 — Demo: Before and After

In [ ]:
SESSION = "bytemonk_demo"

# Session 1 — introduce myself
r1 = run_agent("My name is Himalay. I am a software engineer and founder of ByteMonk.", SESSION)
print(r1)

In [ ]:
# Session 2 — new call, zero history passed in
r2 = run_agent("What do you know about me?", SESSION)
print(r2)

In [ ]:
# Session 3 — semantic search, different words same meaning
r3 = run_agent("Can you remind me what kind of work I do?", SESSION)
print(r3)